# 01 · Supervisado: un árbol de decisión
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrezconz/investigacion-aumentada-ml/blob/main/notebooks/01_supervisado_arbol_titanic.ipynb)

**Investigación Aumentada · Sesión 8 · Machine Learning: herramientas para trabajar con datos**

**Pregunta:** ¿qué características separan a quienes sobrevivieron al Titanic de quienes no?

**Familia:** supervisado. El modelo aprende de 1.309 pasajeros cuyo desenlace ya conocemos (la *respuesta*).

**Cómo funciona un árbol de decisión:** prueba todas las preguntas de sí o no que puede hacer con los datos («¿es mujer?», «¿viajaba en tercera clase?») y escoge la que mejor separa a quienes sobrevivieron de quienes no. Luego repite lo mismo dentro de cada rama.

Para correrlo: menú **Entorno de ejecución → Ejecutar todas**.

In [ ]:
# Preparación: trae los datos del repositorio (funciona igual en Google Colab)
import os, urllib.request
REPO = "https://raw.githubusercontent.com/andrezconz/investigacion-aumentada-ml/main/datos/"

def traer(nombre):
    local = os.path.join("..", "datos", nombre)
    if os.path.exists(local):
        return local
    if not os.path.exists(nombre):
        urllib.request.urlretrieve(REPO + nombre, nombre)
    return nombre

In [ ]:
import pandas as pd
df = pd.read_csv(traer("titanic3.csv"))
df[["pclass", "sex", "age", "fare", "survived"]].head()

`survived` es la respuesta conocida: 1 sobrevivió, 0 no sobrevivió. Esa columna es la que hace que este problema sea **supervisado**.

In [ ]:
# Variables de entrada (X) y respuesta (y)
X = pd.DataFrame({
    "es_mujer": (df["sex"] == "female").astype(int),
    "clase": df["pclass"],
    "edad": df["age"].fillna(df["age"].median()),
    "tarifa": df["fare"].fillna(df["fare"].median()),
})
y = df["survived"]
print(X.shape, "| sobrevivió:", round(y.mean() * 100, 1), "%")

## Guardar una parte para probar
Separamos el 25 % de los pasajeros y el modelo **nunca** los ve mientras aprende. Así podemos saber si sirve con casos nuevos.

> **Tip:** guarda siempre una parte de los datos para probar. Acertar con lo que ya vio no dice nada.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X_ent, X_prueba, y_ent, y_prueba = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

arbol = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_ent, y_ent)
print("Aciertos con lo que vio:   ", round(arbol.score(X_ent, y_ent) * 100, 1), "%")
print("Aciertos con lo que no vio:", round(arbol.score(X_prueba, y_prueba) * 100, 1), "%")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(16, 7))
plot_tree(arbol, feature_names=list(X.columns), class_names=["No sobrevivió", "Sobrevivió"],
          filled=True, rounded=True, fontsize=10, impurity=False, proportion=True)
plt.title("Las preguntas que aprendió el árbol")
plt.show()

## Experimento: ¿qué pasa si dejamos que el árbol crezca sin límite?
Un árbol sin límite puede memorizar a cada pasajero. Mira la diferencia entre lo que vio y lo que no vio: eso se llama **sobreajuste**.

In [ ]:
arbol_gigante = DecisionTreeClassifier(random_state=42).fit(X_ent, y_ent)
print("Árbol sin límite · con lo que vio:   ", round(arbol_gigante.score(X_ent, y_ent) * 100, 1), "%")
print("Árbol sin límite · con lo que no vio:", round(arbol_gigante.score(X_prueba, y_prueba) * 100, 1), "%")
print("Número de hojas:", arbol_gigante.get_n_leaves(), "frente a", arbol.get_n_leaves(), "del árbol pequeño")

## Para seguir con IA
Copia en tu asistente de IA:

> Te paso un árbol de decisión entrenado con datos del Titanic. Explícame en lenguaje sencillo cada pregunta que hace el árbol, por qué crees que la escogió y qué limitaciones tiene usar este modelo para sacar conclusiones causales.

*Fuente de los datos: titanic3, Vanderbilt University, Department of Biostatistics. Elaboración propia.*